# Adult Depression Data Analysis

_A study project by Shaz — polished EDA, baseline models, and publication-ready visualisations for GitHub._

This notebook contains: overview of the dataset, exploratory data analysis, light cleaning and imputation, a baseline classification (Logistic Regression and Decision Tree), evaluation, and suggested next steps.

**Tone:** portfolio-friendly (readable, with clear explanations for recruiters).


## About this project

- **Author:** Shaz
- **Purpose:** Study project to explore adult depression indicators, practice EDA and baseline modelling in Python, and publish results on GitHub.
- **Files used:** `adult-depression-lghc-indicator-24.csv` (original) and `adult_depression_imputed_for_repo.csv` (imputed copy).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Ensure output directories
os.makedirs('/mnt/data/eda_outputs', exist_ok=True)
os.makedirs('/mnt/data/cls_outputs', exist_ok=True)

print('Ready. Files will be saved in /mnt/data/eda_outputs and /mnt/data/cls_outputs')

## 1) Load the imputed dataset

We use the imputed CSV produced earlier (`adult_depression_imputed_for_repo.csv`). If you want to rerun imputation, run the EDA steps in the previous script.

In [ ]:
data_path = '/mnt/data/adult_depression_imputed_for_repo.csv'
df = pd.read_csv(data_path)
print('Loaded dataset shape:', df.shape)
df.head()

## 2) Quick overview: dtypes, missingness, and summary statistics

In [ ]:
display(df.dtypes)

missing = df.isnull().sum().sort_values(ascending=False)
print('\nMissing values per column:')
print(missing)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object','category']).columns.tolist()
print('\nNumeric columns:', num_cols)
print('Categorical columns:', cat_cols)

# Numeric summary
if num_cols:
    display(df[num_cols].describe().transpose())

## 3) Exploratory Data Analysis (plots)

The plots below are styled for readability: larger labels and tight layout. Each plot is also saved to `/mnt/data/eda_outputs/`.

In [ ]:
# Histograms for numeric columns (inline + saved)
for c in num_cols:
    fig, ax = plt.subplots(figsize=(8,4))
    ax.hist(df[c].dropna(), bins=25)
    ax.set_title(f'Histogram — {c}', fontsize=14)
    ax.set_xlabel(c, fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fname = f'/mnt/data/eda_outputs/hist_{c.replace(" ","_")}.png'
    fig.savefig(fname)
    display(fig)
    plt.close(fig)
    print('Saved', fname)

# Bar charts for categorical columns
for c in cat_cols:
    top = df[c].value_counts(dropna=False).head(10)
    fig, ax = plt.subplots(figsize=(10,4))
    ax.bar(range(len(top)), top.values)
    ax.set_xticks(range(len(top.index)))
    ax.set_xticklabels([str(s)[:40] for s in top.index], rotation=45, ha='right')
    ax.set_title(f'Top categories — {c}', fontsize=14)
    ax.set_ylabel('Count', fontsize=12)
    plt.tight_layout()
    fname = f'/mnt/data/eda_outputs/bar_{c.replace(" ","_")}.png'
    fig.savefig(fname)
    display(fig)
    plt.close(fig)
    print('Saved', fname)

# Correlation matrix
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    fig, ax = plt.subplots(figsize=(8,6))
    cax = ax.imshow(corr, interpolation='nearest', aspect='auto')
    ax.set_xticks(range(len(num_cols)))
    ax.set_yticks(range(len(num_cols)))
    ax.set_xticklabels(num_cols, rotation=90)
    ax.set_yticklabels(num_cols)
    for (i, j), val in np.ndenumerate(corr.values):
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9)
    ax.set_title('Correlation matrix (numeric)', fontsize=14)
    fig.colorbar(cax, ax=ax)
    plt.tight_layout()
    fname = '/mnt/data/eda_outputs/correlation_matrix.png'
    fig.savefig(fname)
    display(fig)
    plt.close(fig)
    print('Saved', fname)
else:
    print('Not enough numeric columns for correlation matrix.')

## 4) Baseline classification

We create a simple binary target `HighDepression` defined as `Percent > median(Percent)`. This choice is documented and can be changed. We then train a Logistic Regression and a Decision Tree (max_depth=4) as baselines.

In [ ]:
# Create binary target
median_percent = df['Percent'].median()
df['HighDepression'] = (df['Percent'] > median_percent).astype(int)
print('Median Percent:', median_percent)
print('\nTarget distribution:')
print(df['HighDepression'].value_counts())

# Features: exclude Percent and the target
exclude = ['Percent','HighDepression']
feature_cols = [c for c in df.columns if c not in exclude]
X = df[feature_cols].copy()
y = df['HighDepression'].copy()

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object','category']).columns.tolist()
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Preprocessor and pipelines
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), categorical_features)
], remainder='drop')

log_pipe = Pipeline(steps=[('pre', preprocessor), ('clf', LogisticRegression(max_iter=1000, solver='liblinear'))])
tree_pipe = Pipeline(steps=[('pre', preprocessor), ('clf', DecisionTreeClassifier(max_depth=4, random_state=42))])

# Fit
log_pipe.fit(X_train, y_train)
tree_pipe.fit(X_train, y_train)
print('Models trained.')

## 5) Evaluation

We report accuracy, precision, recall, F1 and ROC AUC. Confusion matrices and ROC curves are plotted and saved.

In [ ]:
models = {'Logistic Regression': log_pipe, 'Decision Tree (max_depth=4)': tree_pipe}
results = []

for name, pipe in models.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:,1]
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_test, y_proba)
    except Exception:
        auc = np.nan
    results.append({'model': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'roc_auc': auc})
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5,4))
    im = ax.imshow(cm, interpolation='nearest')
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, int(val), ha='center', va='center', fontsize=12)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['pred_0','pred_1']); ax.set_yticklabels(['actual_0','actual_1'])
    ax.set_title(f'Confusion Matrix — {name}', fontsize=12)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    cm_fname = f'/mnt/data/cls_outputs/confusion_matrix_{name.replace(" ","_")}.png'
    fig.savefig(cm_fname)
    display(fig)
    plt.close(fig)
    print('Saved', cm_fname)
    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot(fpr, tpr)
    ax.plot([0,1],[0,1], linestyle='--')
    ax.set_title(f'ROC Curve — {name}', fontsize=14)
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.grid(True)
    plt.tight_layout()
    roc_fname = f'/mnt/data/cls_outputs/roc_{name.replace(" ","_")}.png'
    fig.savefig(roc_fname)
    display(fig)
    plt.close(fig)
    print('Saved', roc_fname)

results_df = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
display(results_df)
results_df.to_csv('/mnt/data/cls_outputs/model_comparison_summary.csv', index=False)
print('Saved model comparison summary to /mnt/data/cls_outputs/model_comparison_summary.csv')

## 6) Decision Tree visualization & feature names

We extract feature names after preprocessing and plot the tree (saved to cls_outputs).

In [ ]:
# Extract feature names after preprocessing
pre = preprocessor.fit(X_train)
num_names = numeric_features
cat_transformer = pre.named_transformers_['cat']
if hasattr(cat_transformer, 'get_feature_names_out'):
    cat_names = list(cat_transformer.get_feature_names_out(categorical_features))
else:
    cat_names = []
feature_names = num_names + cat_names

# Plot tree
fig, ax = plt.subplots(figsize=(14,8))
tree_clf = tree_pipe.named_steps['clf']
plot_tree(tree_clf, feature_names=feature_names, class_names=['low','high'], filled=False, fontsize=8, ax=ax)
plt.tight_layout()
tree_fname = '/mnt/data/cls_outputs/decision_tree_visualization.png'
fig.savefig(tree_fname)
display(fig)
plt.close(fig)
print('Saved', tree_fname)

# Feature importances (tree)
importances = getattr(tree_clf, 'feature_importances_', None)
if importances is not None and len(importances) == len(feature_names):
    imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
    display(imp_df.head(20))
    imp_df.to_csv('/mnt/data/cls_outputs/decision_tree_feature_importance.csv', index=False)
    print('Saved feature importance to /mnt/data/cls_outputs/decision_tree_feature_importance.csv')
else:
    print('Could not extract feature importances.')

## 7) Conclusions & next steps

- This notebook provides a reproducible EDA and baseline modelling workflow. The `HighDepression` target was defined as `Percent > median(Percent)` (median value shown above).
- The models are intentionally simple baselines to demonstrate the pipeline. Recommended next steps:
  - Cross-validated performance and hyperparameter tuning (GridSearchCV).
  - Feature selection and explainability (SHAP) for interpretability.
  - If you have a clear target variable or labels, replace the thresholded target and retrain.

---

**To include in your GitHub repo:**
- This notebook file: `polished_adult_depression_notebook.ipynb`
- The `eda_outputs/` and `cls_outputs/` folders (images + CSVs) stored alongside the notebook.

Feel free to ask me to (1) change the target threshold (e.g., 75th percentile), (2) run GridSearch and CV, or (3) produce an attractive README.md summarising findings for recruiters.